# Project Title: Finding an Unbiased Pendulum Model for Parameter-Estimation Simulations

In [12]:
NAMES = ['Angela DiAndreth', 'Sergey Dutt', 'Edward Lombo-Becerra']

# Introduction:

Many common engineering components move according to simple harmonic motion (pendulums, springs, strings, and wires). When designing systems with these parts, you often need their oscillations to decay at a specific rate. To aid in hardware sizing, you can input this desired decay target into a parameter simulation, which directly outputs physical specifications like length and mass.

Here is an example problem that we created: Consider an inertia-driven semi-automatic shotgun used in trap and skeet competitions. In this system, the bolt body acts as a floating mass (m), and an inertia spring of length L sits compressed between the bolt body and the bolt carrier. Firing a standard 12-gauge shell imparts an average force of 1,250N into the spring, pushing the bolt carrier backward to unlock the action, eject the spent shell, and chamber a new round.
To achieve a target fire rate of 4 rounds per second (240RPM) without jamming or misfiring, the spring’s oscillation must decay by $99\%$ before the next shot. Based on this fire rate and safety margin, the spring requires a decay constant of at least 36.84s-1. Additionally, the inertia spring can be no longer than 8cm and the bolt no heavier than 500g. The simulation must determine the optimal spring length ($L$), bolt mass ($m$), and spring stiffness ($k$) to meet these goals.

For this simulation to work, the math model must rely on unbiased physical parameters. This means any uncertainty in a parameter like length ($L$) must come purely from random noise, rather than absorbing the unaccounted physics of another variable. The goal of this project is to use experimental pendulum data to calibrate a simple harmonic model that accurately tracks trajectory data while keeping its physical parameters completely unbiased. If we are successful in finding a model that matches both criteria, we may attempt the example problem from the previous paragraph.

The differential equation for simple harmonic motion takes the standard form (Shankar, 1994):

$$
\ddot{x} + a_1 \dot{x} + a_2 x = 0 \tag{1} 
$$

with the analytical solution:

$$
x(t) = e^{-\alpha t} \left( A \cos(\beta t) + B \sin(\beta t) \right) \tag{2} 
$$

This following is how the differential equation for a pendulum decaying due to linear air resistance takes the form of simple harmonic motion. Beginning with the Euler-Lagrange equation derived from Hamilton's Principle of Least Action in classical mechanics (Shankar, 1994):

$$
\frac{d}{dt} \left( \frac{\partial \mathcal{L}}{\partial \dot{q}_j} \right) - \frac{\partial \mathcal{L}}{\partial q_j} = 0 \tag{3} 
$$

The Lagrangian ($\mathcal{L}$) represents the total kinetic energy ($T$) minus the total potential energy ($U$) of the system:
$$
\mathcal{L} = T - U \tag{4} 
$$

In Cartesian coordinates, $T = \frac{1}{2} m v^2$ and $U = mgh$. For a pendulum, its height relative to its lowest point is $h = L(1 - \cos\theta)$, where $\theta$ is the angular displacement from equilibrium. Similarly, its linear velocity $v$ is the time derivative of its arc length $s = L \theta$, yielding $v = L \dot{\theta}$.

Plugging these into the Lagrangian yields:

$$
\mathcal{L} = \frac{1}{2} m L^2 \dot{\theta}^2 + mgL(1 - \cos\theta) \tag{5} 
$$

However, the conservative Lagrangian formulation only accounts for restorative forces and neglects energy loss. To model the motion of our decaying pendulum, we introduce non-conservative dissipation by replacing the right-hand side of the Euler-Lagrange equation with a linear Rayleigh drag force:

$$
\frac{d}{dt} \left( \frac{\partial \mathcal{L}}{\partial \dot{\theta}} \right) - \frac{\partial \mathcal{L}}{\partial \theta} = -c \dot{\theta} \tag{6} 
$$

Evaluating the derivatives gives:

$$
m L^2 \ddot{\theta} + c \dot{\theta} + mgL \sin\theta = 0 \tag{7} 
$$

Dividing by the rotational inertia term $m L^2$ puts the model into standard form:

$$
\ddot{\theta} + a_1 \dot{\theta} + a_2 \sin\theta = 0 \tag{8} 
$$

where:

$$
a_1 = \frac{c}{m L^2}, \quad a_2 = \frac{g}{L} \tag{9, 10} 
$$

After applying the small-angle Taylor series approximation $\sin\theta \approx \theta$, we arrive directly at the differential equation for simple harmonic motion Eq. (2).

# Methods:

In [13]:
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from jax import jit

import numpy as np
import scipy.signal as signal
from scipy.optimize import minimize_scalar
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display
from scipy.integrate import solve_ivp

np.random.seed(42)  # for reproducibility

Pendulum.xlsx contains experimental pendulum motion data collected from an undergraduate physics project at Luther College. The angular position relative to the equilibrium position was recorded at an interval of 0.1 seconds over approximately one minute. The workbook contains multiple sheets corresponding to different wire thicknesses holding the pendulum mass.

Since the mathematical models assume that the equillibrium is at $\theta = 0$ and the recorded data in `Pendulum.xlsx` off-centered, `load_and_center_data` realigns the equilibrium point across the file. Additionally, we will use subsets of the full dataset because the spreadsheet is very large (4936 datapoints). `get_random_subset` selects a random group of points that starts at a local max and meets a specified oscillation requirement. 

In [21]:
from fix_data import get_random_subset, load_and_center_data

In [22]:
df = load_and_center_data("Pendulum.xlsx")
total_peaks = 20
train_peaks = 10
subset_df = get_random_subset(df, total_peaks) 

original_t_start = subset_df["Time"].iloc[0]
original_t_end = subset_df["Time"].iloc[-1]

print(f"Original Subset Time Window: {original_t_start:.1f} s to {original_t_end:.1f} s\
")

def get_t_and_theta(t, theta):
    # start subset at t = 0                                                             
    subset_df[t] = subset_df[t] - subset_df[t].iloc[0]

    x = np.array(subset_df[t].values)
    y = np.array(subset_df[theta].values)

    print(f"Number of Data Points {len(x):.1f}")

    return x, y

t_full, theta_full = get_t_and_theta(t = "Time", theta = "Angle")

peaks, _ = signal.find_peaks(theta_full)
split = peaks[train_peaks]

t_data = t_full[:split]
theta_data = theta_full[:split]
t_test = t_full[split:]
theta_test = theta_full[split:]

print(f"Train Set: {len(t_data)} pts ({t_data[0]:.2f}s to {t_data[-1]:.2f}s)")
print(f"Test Set: {len(t_test)} pts ({t_test[0]:.2f}s to {t_test[-1]:.2f}s)")

Original Subset Time Window: 243.4 s to 295.7 s
Number of Data Points 524.0
Train Set: 288 pts (0.00s to 28.70s)
Test Set: 236 pts (28.80s to 52.30s)


This project determines the optimal physical parameters $a_1$ and $a_2$ using linear regression. Given $m$ data points $(t_m, \theta_m)$, the matrix formulation is given by:

$$
\begin{bmatrix} \ddot{\theta}(t_1) \\ \vdots \\ \ddot{\theta}(t_m) \end{bmatrix} = \begin{bmatrix} -\dot{\theta}(t_1) & -\sin(\theta(t_1)) \\ \vdots & \vdots \\ -\dot{\theta}(t_m) & -\sin(\theta(t_m)) \end{bmatrix} \begin{bmatrix} a_1 \\ a_2 \end{bmatrix} \tag{11}
$$

Which we will refer to in our code as:

$$
Y = Xa \tag{12}
$$

Because only discrete measurements of $\theta$ are available, we first reconstruct a continuous, smooth, and differentiable representation of the trajectory. Once established, automatic differentiation (via JAX) can be applied directly to compute the time derivatives $\dot{\theta}(t)$ and $\ddot{\theta}(t)$. Alternatively, we could estimate $a_1$ and $a_2$ using nonlinear least-squares optimization. However, because Eq. (8) lacks a closed-form analytical solution, and is inherently nonlinear in $\theta$ even in the undamped case Eq. (2), nonlinear optimization requires repeated numerical integration and good initial parameter guesses. By leveraging automatic differentiation to evaluate the derivatives directly, we bypass iterative optimization and initial guesses altogether, allowing $a_1$ and $a_2$ to be solved efficiently via standard linear regression.

In order to use linear regression on Eq. (11), the data is first modeled as a differentiable Fourier Series that is similar in form to the simple harmonic oscillator: 

$$
\theta(t) = e^{-\alpha t} \sum_{k=1}^{\infty} \left[ A_k \cos(k \beta t) + B_k \sin(k \beta t) \right] \tag{13}
$$

For $m$ data points truncated at $N$ iterations, the coefficients $(A_k, B_k)$ are fitted using the linear system:

$$\begin{bmatrix} \theta(t_1) \\ \vdots \\ \theta(t_m) \end{bmatrix} = \begin{bmatrix} e^{-\alpha t_1}\cos(1\beta t_1) & e^{-\alpha t_1}\sin(1\beta t_1) & \dots & e^{-\alpha t_1}\cos(N\beta t_1) & e^{-\alpha t_1}\sin(N\beta t_1) \\ \vdots & \vdots & \ddots & \vdots & \vdots \\ e^{-\alpha t_m}\cos(1\beta t_m) & e^{-\alpha t_m}\sin(1\beta t_m) & \dots & e^{-\alpha t_m}\cos(N\beta t_m) & e^{-\alpha t_m}\sin(N\beta t_m) \end{bmatrix} \begin{bmatrix} A_1 \\ B_1 \\ \vdots \\ A_N \\ B_N \end{bmatrix} \tag{14}
$$

To prevent overfitting, model accuracy is evaluated using the Mean Squared Error (MSE):
$$
\text{MSE} = \frac{1}{m} \sum_{i=1}^{m} \left( \theta_{\text{data}}(t_i) - \theta(t_i) \right)^2 \tag{14}
$$

We stop adding terms to the series when the relative improvement in MSE drops below a pre-set tolerance threshold:

$$
\frac{\text{MSE}(N-1) - \text{MSE}(N)}{\text{MSE}(N-1)} < \text{tol}, \quad \text{where } \text{tol} = 0.01\text{ (1\%)} \tag{15}
$$

In [25]:
def f_series(t_data, theta_data,
             K_max = 10, tol = 0.01,
             search_for_freq_opt=True, freq_opt=None):

    peaks, _ = signal.find_peaks(theta_data)

    t_diff = t_data[peaks[-1]] - t_data[peaks[0]]
    avg_period = t_diff / (len(peaks) - 1)
    beta_guess = 2 * np.pi / avg_period

    first_peak = theta_data[peaks[0]]
    last_peak = theta_data[peaks[-1]]
    alpha = np.log(first_peak / last_peak) / t_diff
    decay = np.exp(-alpha * t_data)

    def lin_reg(beta, K):
        cols = []
        for k in range(1, K + 1):
            cols.append(decay * np.cos(k * beta * t_data))
            cols.append(decay * np.sin(k * beta * t_data))

        X = np.column_stack(cols)
        a = np.linalg.solve(X.T @ X, X.T @ theta_data)
        theta_model = X @ a
        mse = np.mean((theta_data - theta_model) ** 2)
        return a, mse

    # Mode 1: Find Optimal Frequency
    if search_for_freq_opt:
        res = minimize_scalar(
            lambda beta: lin_reg(beta, K_max)[1],
            bounds = (beta_guess * 0.95, beta_guess * 1.05),
            method = "bounded",
        )
        freq_opt = res.x
        print(f"Optimal Frequency is: {freq_opt:.3f}")
        return freq_opt

    # Mode 2: Fit Fourier Series using freq_opt
    for K in  range(1, K_max + 1):
        a_k, MSE = lin_reg(freq_opt, K)

        if K > 1:
            rel_improvement = (prev_MSE - MSE) / prev_MSE
            if rel_improvement < tol:
                break

        a = a_k
        K_total = K
        prev_MSE = MSE

    return Lambda, a, K_total   

Add desciption of why we use minimize_scalar and how we approximate alpha and beta

In [31]:
freq_opt = f_series(t_data, theta_data)

Lambda, a, K_total = f_series(
    t_data = t_data,
    theta_data = theta_data,
    search_for_freq_opt = False,
    freq_opt = freq_opt,
)

print(f"Number of cos/sin combinations in the Fourier Series is k = {K_total}")

Optimal Frequency is: 2.400
Number of cos/sin combinations in the Fourier Series is k = 3


In [27]:
a = jnp.array(a)
def theta_func(t_m): # m data points (t_m, theta_m)                                        
    decay_t = jnp.exp(-Lambda * t_m)
    terms = jnp.array([
        a[2*(k-1)] * jnp.cos(k * freq_opt * t_m) +
        a[2*(k-1)+1] * jnp.sin(k * freq_opt * t_m)
        for k in range(1, K_total + 1)
    ])
    return decay_t * jnp.sum(terms)

@jit # for fast computation of jax.grad(jax.grad())                                        
def get_derivatives(t):

    dtheta_dt = jax.vmap(jax.grad(theta_func))(t)
    d2theta_dt2 = jax.vmap(jax.grad(jax.grad(theta_func)))(t)

    return dtheta_dt, d2theta_dt2

theta_dot, theta_ddot = map(np.asarray, get_derivatives(jnp.asarray(t_data)))

give equation for residuals, and how we use them to estimate the uncertainty on the optimal paramater values

In [28]:
# linear regression againts pendulum model: Xa = Y                                         
# physics model: x'' = -a1 * x' - a2 * sin(x), x = theta                                   
Y = theta_ddot
X = np.column_stack([-theta_dot, -np.sin(theta_data)])

a = np.linalg.solve(X.T @ X, X.T @ Y)
a1 = float(a[0])
a2 = float(a[1])

# Residuals                                                                                
residuals = Y - X @ a

ss_res = np.sum(residuals**2)
ss_tot = np.sum((Y - np.mean(Y))**2)
r_squared = 1 - (ss_res / ss_tot)

N, p = X.shape # N data points, p = 2 parameters                                           

sigma2 = np.sum(residuals ** 2) / (N - p)
cov_matrix = sigma2 * np.linalg.inv(X.T @ X)

std_errors = np.sqrt(np.diag(cov_matrix))
se_a1, se_a2 = std_errors[0], std_errors[1]

print(f"Optimal a1: {a1:.5f} +/- {se_a1:.5f} ")
print(f"Optimal a2: {a2:.5f} +/- {se_a2:.5f} ")

Optimal a1: 0.00642 +/- 0.00327 
Optimal a2: 5.80882 +/- 0.00792 


In [29]:
# a1 = c / m*L^2, a2 = g/L                                                              
g = 9.81 # m / s^2                                                                      
m_data = 1 # kg                                                                         
L_model = g / a2
C_model = a1 * m_data * (L_model**2)

print(f"L_model: {L_model:.3f} m")
print(f'Air-Resistance Coef (C): {C_model:.3f}')

L_model: 1.689 m
Air-Resistance Coef (C): 0.018


describe the differential equation in terms of a vector space of orthogonal basis vectors

In [30]:
# prove that uncertainties in a1 and a2 are uncorrelated                                   
# (if noise causes error in a1, does it also cause error in a2?)                           
cov_a1_a2 = cov_matrix[0, 1] # off diagonal: Cov(a1, a2)                                   
corr_a1_a2 = cov_a1_a2 / (se_a1 * se_a2) # unitless Cov(a1, a2)                            
print(f"The correlation (normalized Cov) between a1 and a2 is {corr_a1_a2:.3e}")

col1, col2 = X[:, 0], X[:, 1] # columns of X matrix in Y = Xa 
signal_1 = a1 * col1
signal_2 = a2 * col2

# how much of the signal in col1 leaks into the signal in col2                             
col1_norm = np.linalg.norm(col1)
col2_norm = np.linalg.norm(col2)
col1_col2_proj = np.dot(col1, col2) / (col1_norm * col2_norm)
print(f"The projection of Col. 1 onto Col. 2 (from matrix X) is: {col1_col2_proj:.3e}")

# this will always be zero (~1e-15)                                          
res_norm = np.linalg.norm(residuals)
res_col1_proj = np.dot(residuals, col1) / (res_norm * col1_norm)
res_col2_proj = np.dot(residuals, col2) / (res_norm * col2_norm)
print(f"The projection of residuals onto Col. 1 is: {res_col1_proj:.1e}")
print(f"The projection of residuals onto Col. 2 is: {res_col2_proj:.1e}")

The correlation (normalized Cov) between a1 and a2 is 2.689e-03
The projection of Col. 1 onto Col. 2 (from matrix X) is: -2.689e-03
The projection of residuals onto Col. 1 is: -1.2e-15
The projection of residuals onto Col. 2 is: 2.6e-14


# Figures: